# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbinaqeel-analyst/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# ============================================================
# WEEK 6 — Hugging Face access check
# ============================================================

import os
import duckdb

# 1. Check whether the Colab secret exists.
# Never print the actual token.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. Add your existing Hugging Face READ token "
        "to Colab Secrets as HF_TOKEN."
    )

print("HF_TOKEN: loaded")
print("Token length:", len(HF_TOKEN))

# 2. Create DuckDB connection.
con = duckdb.connect()

# 3. Authenticate Hugging Face.
con.execute("DROP SECRET IF EXISTS hf_secret")
con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
""")

print("DuckDB: connected")
print("Hugging Face: authentication configured")

HF_TOKEN: loaded
Token length: 37
DuckDB: connected
Hugging Face: authentication configured


## 1. Two paper findings + my methodology questions

### Finding 1 — The Content Performance Curve

The paper reports that content health is highest around 61–90 days, declines after 270 days, and shows a higher 365+ result that is concentrated in older pages that were refreshed.

**Methodology question:** What exactly defines the outcome being compared across these age bands, and is the health score measured from the same observation window for every band? I would also want to confirm that the 365+ result is not being driven by the small number of pages in that group or by pages that were already selected for refresh.

This is a useful descriptive finding, but the validation question is whether the comparison supports a statement about an observed relationship between content age, refresh status, and measured performance, rather than a causal claim that age or refreshing produced the observed change.*

### Finding 2 — Click Capture by Position Tier

The paper reports that weighted CTR decreases as pages move farther from the top search positions: 0.423% for the top 3, 0.339% for positions 4–10, 0.325% for positions 11–20, 0.163% for positions 21–50, and 0.050% for deeper positions.

**Methodology question:** Are these position tiers being evaluated using a common observation window and weighted by total impressions, and how are pages with very different impression volumes handled? I would also ask whether the result is intended only as a portfolio-level descriptive relationship rather than evidence that moving a page to a higher position would itself cause the CTR increase.

The paper addresses part of this by stating that these are portfolio-level weighted CTRs rather than generic Google CTR benchmarks. The remaining validation question is how far the observed association can reasonably be generalized beyond this portfolio and reporting window.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

### Validation design

The Week-5 model used a client-holdout design on the starter slice. For this audit, I compare that result with a stricter time-aware evaluation.

The time-aware design is appropriate because the intended research question concerns content movement over time. Training data should therefore precede the evaluation period rather than mixing observations from the same overall period.

The comparison uses the same target definition and the same model family so that the main change is the validation design.

The target remains the Week-5 proxy label:

`is_declining_label = trend_direction == "down"`

This is an important limitation: the label describes the current observed trend window rather than a clean future outcome. Therefore the validation results measure performance against this proxy, not the ability to predict a future decline after a decision point.

In [3]:
# ============================================================
# SECTION 2 - SETUP AND DATA LOADING
# ============================================================

import os
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. Expected paths
# ------------------------------------------------------------

REPO_PATH = "/content/flyrank-ml-internship"
STARTER_PATH = os.path.join(
    REPO_PATH,
    "data",
    "raw",
    "content_refresh_anonymized.csv"
)

# ------------------------------------------------------------
# 2. If the repository is not present in this Colab runtime,
#    clone it from GitHub.
# ------------------------------------------------------------

if not os.path.exists(STARTER_PATH):

    print("Starter CSV not found at:")
    print(STARTER_PATH)
    print()
    print("Checking whether the GitHub repository exists locally...")

    if not os.path.exists(REPO_PATH):

        print("Repository not found locally.")
        print("Cloning GitHub repository...")

        result = subprocess.run(
            [
                "git",
                "clone",
                "https://github.com/mbinaqeel-analyst/flyrank-ml-internship.git",
                REPO_PATH
            ],
            capture_output=True,
            text=True
        )

        if result.returncode != 0:
            print("Git clone failed.")
            print(result.stderr)
            raise RuntimeError(
                "Could not clone the GitHub repository. "
                "Check the repository URL/access and try again."
            )

        print("Repository cloned successfully.")

# ------------------------------------------------------------
# 3. Final file check
# ------------------------------------------------------------

if not os.path.exists(STARTER_PATH):
    raise FileNotFoundError(
        f"\nStarter CSV still not found:\n{STARTER_PATH}\n\n"
        "Check that content_refresh_anonymized.csv exists under "
        "data/raw/ in the repository."
    )

print("Starter file found:")
print(STARTER_PATH)

# ------------------------------------------------------------
# 4. Load data
# ------------------------------------------------------------

df = pd.read_csv(STARTER_PATH)

print("\nShape:", df.shape)
print("Columns:", len(df.columns))

display(df.head())

Starter CSV not found at:
/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv

Checking whether the GitHub repository exists locally...
Repository not found locally.
Cloning GitHub repository...
Repository cloned successfully.
Starter file found:
/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv

Shape: (30000, 44)
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [4]:
# ============================================================
# MODELING POPULATION
# ============================================================

# Make a working copy
model_df = df.copy()

# ------------------------------------------------------------
# Target
# ------------------------------------------------------------

if "trend_direction" not in model_df.columns:
    raise KeyError(
        "Expected target column 'trend_direction' was not found."
    )

model_df["is_declining_label"] = (
    model_df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

# ------------------------------------------------------------
# Apply the same population restrictions used for the model:
# - pages with impressions
# - content at least 90 days old
# ------------------------------------------------------------

if "impressions_90d" not in model_df.columns:
    raise KeyError("Column 'impressions_90d' not found.")

if "content_age_days" not in model_df.columns:
    raise KeyError("Column 'content_age_days' not found.")

model_df = model_df[
    (model_df["impressions_90d"] > 0)
    &
    (model_df["content_age_days"] >= 90)
].copy()

# ------------------------------------------------------------
# Remove duplicate content IDs when available
# ------------------------------------------------------------

if "content_id" in model_df.columns:
    before_dedup = len(model_df)

    model_df = (
        model_df
        .drop_duplicates(subset=["content_id"])
        .copy()
    )

    print("Rows removed by content_id deduplication:",
          before_dedup - len(model_df))

# ------------------------------------------------------------
# Basic target diagnostics
# ------------------------------------------------------------

print("Modeling rows:", len(model_df))
print()

print("Target distribution:")
print(
    model_df["is_declining_label"]
    .value_counts()
    .sort_index()
)

print()

print("Target proportions:")
print(
    model_df["is_declining_label"]
    .value_counts(normalize=True)
    .sort_index()
)

Rows removed by content_id deduplication: 0
Modeling rows: 30000

Target distribution:
is_declining_label
0    13738
1    16262
Name: count, dtype: int64

Target proportions:
is_declining_label
0    0.457933
1    0.542067
Name: proportion, dtype: float64


In [5]:
# ============================================================
# FEATURE DEFINITION
# ============================================================

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

# ------------------------------------------------------------
# Keep only features that actually exist
# ------------------------------------------------------------

missing_numeric = [
    col for col in numeric_features
    if col not in model_df.columns
]

missing_categorical = [
    col for col in categorical_features
    if col not in model_df.columns
]

if missing_numeric:
    print("Missing numeric features:")
    print(missing_numeric)

if missing_categorical:
    print("\nMissing categorical features:")
    print(missing_categorical)

# Stop if required features are absent
if missing_numeric or missing_categorical:
    raise KeyError(
        "One or more Week-5 model features are missing from the dataset."
    )

all_features = numeric_features + categorical_features

X = model_df[all_features].copy()
y = model_df["is_declining_label"].copy()

print("Number of features:", len(all_features))
print("X shape:", X.shape)
print("y shape:", y.shape)

Number of features: 26
X shape: (30000, 26)
y shape: (30000,)


In [6]:
# ============================================================
# CLIENT GROUP CHECK
# ============================================================

possible_client_columns = [
    "client_hash_id",
    "client_id",
    "client"
]

client_column = None

for col in possible_client_columns:
    if col in model_df.columns:
        client_column = col
        break

if client_column is None:
    raise KeyError(
        "No client identifier column was found. "
        "A client-grouped validation split requires a client column."
    )

groups = model_df[client_column].astype(str)

print("Client column:", client_column)
print("Unique clients:", groups.nunique())
print()
print("Rows per client:")
display(groups.value_counts().describe())

Client column: client_id
Unique clients: 32

Rows per client:


,count
count,32.000000
mean,937.500000
std,1376.387113
min,3.000000
25%,110.250000
50%,567.000000
75%,1058.750000
max,7008.000000


In [7]:
# ============================================================
# BEFORE: CLIENT-GROUPED HOLDOUT
# ============================================================

from sklearn.model_selection import GroupShuffleSplit

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score
)

# ------------------------------------------------------------
# Client-grouped split
# ------------------------------------------------------------

group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    group_split.split(
        X,
        y,
        groups=groups
    )
)

X_train_before = X.iloc[train_idx].copy()
X_test_before = X.iloc[test_idx].copy()

y_train_before = y.iloc[train_idx].copy()
y_test_before = y.iloc[test_idx].copy()

groups_train_before = groups.iloc[train_idx]
groups_test_before = groups.iloc[test_idx]

print("BEFORE - Client grouped split")
print("--------------------------------")
print("Train rows:", len(X_train_before))
print("Test rows:", len(X_test_before))
print(
    "Train clients:",
    groups_train_before.nunique()
)
print(
    "Test clients:",
    groups_test_before.nunique()
)

print()
print(
    "Client overlap:",
    len(
        set(groups_train_before)
        &
        set(groups_test_before)
    )
)

BEFORE - Client grouped split
--------------------------------
Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7

Client overlap: 0


In [8]:
# ============================================================
# BEFORE: RANDOM FOREST MODEL
# ============================================================

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

rf_model_before = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample",
    min_samples_leaf=2
)

pipeline_before = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", rf_model_before)
    ]
)

pipeline_before.fit(
    X_train_before,
    y_train_before
)

# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

y_prob_before = pipeline_before.predict_proba(
    X_test_before
)[:, 1]

y_pred_before = (
    y_prob_before >= 0.50
).astype(int)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

roc_before = roc_auc_score(
    y_test_before,
    y_prob_before
)

ap_before = average_precision_score(
    y_test_before,
    y_prob_before
)

precision50_before = precision_score(
    y_test_before,
    y_pred_before,
    zero_division=0
)

print("BEFORE - Client-grouped validation")
print("-----------------------------------")
print(f"ROC AUC:       {roc_before:.3f}")
print(f"Average Prec.: {ap_before:.3f}")
print(f"Precision@0.50:{precision50_before:.3f}")

BEFORE - Client-grouped validation
-----------------------------------
ROC AUC:       0.619
Average Prec.: 0.611
Precision@0.50:0.588


In [9]:
# ============================================================
# BEFORE: RANDOM FOREST MODEL
# ============================================================

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

rf_model_before = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample",
    min_samples_leaf=2
)

pipeline_before = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", rf_model_before)
    ]
)

pipeline_before.fit(
    X_train_before,
    y_train_before
)

# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

y_prob_before = pipeline_before.predict_proba(
    X_test_before
)[:, 1]

y_pred_before = (
    y_prob_before >= 0.50
).astype(int)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

roc_before = roc_auc_score(
    y_test_before,
    y_prob_before
)

ap_before = average_precision_score(
    y_test_before,
    y_prob_before
)

precision50_before = precision_score(
    y_test_before,
    y_pred_before,
    zero_division=0
)

print("BEFORE - Client-grouped validation")
print("-----------------------------------")
print(f"ROC AUC:       {roc_before:.3f}")
print(f"Average Prec.: {ap_before:.3f}")
print(f"Precision@0.50:{precision50_before:.3f}")

BEFORE - Client-grouped validation
-----------------------------------
ROC AUC:       0.619
Average Prec.: 0.611
Precision@0.50:0.588


In [11]:
# ============================================================
# AFTER: TIME-AWARE VALIDATION AVAILABILITY CHECK
# ============================================================

date_candidates = [
    "report_date",
    "date",
    "snapshot_date",
    "period",
    "month"
]

available_date_columns = [
    col for col in date_candidates
    if col in model_df.columns
]

print("Candidate observation-date columns found:")
print(available_date_columns)

print()

if not available_date_columns:
    print(
        "No genuine observation-date column is available "
        "in the starter dataset."
    )
    print()
    print(
        "The available date-like field is "
        "'days_since_last_update', but this is a page-level "
        "feature rather than an observation date."
    )
    print()
    print(
        "Therefore, a genuine time-aware split cannot be "
        "performed on this starter dataset."
    )
else:
    print(
        "A possible observation-date column is available:",
        available_date_columns
    )

Candidate observation-date columns found:
[]

No genuine observation-date column is available in the starter dataset.

The available date-like field is 'days_since_last_update', but this is a page-level feature rather than an observation date.

Therefore, a genuine time-aware split cannot be performed on this starter dataset.


In [12]:
# ============================================================
# AFTER: SECOND CLIENT-GROUPED HOLDOUT
# ============================================================
#
# Because the starter CSV has no observation date, we cannot
# honestly construct a time-aware split.
#
# Instead, we use a second independent client-grouped holdout
# with a different random seed.
#
# This tests whether the Week-5 result is sensitive to the
# particular client partition used.
# ============================================================

from sklearn.model_selection import GroupShuffleSplit

group_split_after = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=2026
)

train_idx_after, test_idx_after = next(
    group_split_after.split(
        X,
        y,
        groups=groups
    )
)

X_train_after = X.iloc[train_idx_after].copy()
X_test_after = X.iloc[test_idx_after].copy()

y_train_after = y.iloc[train_idx_after].copy()
y_test_after = y.iloc[test_idx_after].copy()

groups_train_after = groups.iloc[train_idx_after]
groups_test_after = groups.iloc[test_idx_after]

print("AFTER - Independent client-grouped holdout")
print("--------------------------------------------")

print("Train rows:", len(X_train_after))
print("Test rows:", len(X_test_after))

print(
    "Train clients:",
    groups_train_after.nunique()
)

print(
    "Test clients:",
    groups_test_after.nunique()
)

print()

print(
    "Client overlap:",
    len(
        set(groups_train_after)
        &
        set(groups_test_after)
    )
)

print()

print("Train target rate:")
print(y_train_after.mean())

print()

print("Test target rate:")
print(y_test_after.mean())

AFTER - Independent client-grouped holdout
--------------------------------------------
Train rows: 26119
Test rows: 3881
Train clients: 25
Test clients: 7

Client overlap: 0

Train target rate:
0.5320647804280408

Test target rate:
0.6093790260242206


In [13]:
# ============================================================
# AFTER: RANDOM FOREST ON INDEPENDENT CLIENT HOLDOUT
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# ------------------------------------------------------------
# Preprocessing
# ------------------------------------------------------------

numeric_transformer_after = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

categorical_transformer_after = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

preprocessor_after = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer_after,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer_after,
            categorical_features
        )
    ]
)

# ------------------------------------------------------------
# Same Random Forest specification as before
# ------------------------------------------------------------

rf_model_after = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample",
    min_samples_leaf=2
)

pipeline_after = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_after
        ),
        (
            "model",
            rf_model_after
        )
    ]
)

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

pipeline_after.fit(
    X_train_after,
    y_train_after
)

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

y_prob_after = pipeline_after.predict_proba(
    X_test_after
)[:, 1]

y_pred_after = (
    y_prob_after >= 0.50
).astype(int)

print("Model trained successfully.")

Model trained successfully.


In [14]:
# ============================================================
# AFTER: EVALUATION
# ============================================================

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score
)

roc_after = roc_auc_score(
    y_test_after,
    y_prob_after
)

ap_after = average_precision_score(
    y_test_after,
    y_prob_after
)

precision50_after = precision_score(
    y_test_after,
    y_pred_after,
    zero_division=0
)

print("AFTER - Independent client-grouped validation")
print("-----------------------------------------------")
print(f"ROC AUC:         {roc_after:.3f}")
print(f"Average Precision:{ap_after:.3f}")
print(f"Precision @ 0.50:{precision50_after:.3f}")

AFTER - Independent client-grouped validation
-----------------------------------------------
ROC AUC:         0.652
Average Precision:0.717
Precision @ 0.50:0.678


In [15]:
# ============================================================
# BEFORE vs AFTER COMPARISON
# ============================================================

comparison = pd.DataFrame({
    "Validation": [
        "Before: client-grouped (seed=42)",
        "After: client-grouped (seed=2026)"
    ],
    "ROC AUC": [
        roc_before,
        roc_after
    ],
    "Average Precision": [
        ap_before,
        ap_after
    ],
    "Precision @ 0.50": [
        precision50_before,
        precision50_after
    ]
})

display(
    comparison.style.format({
        "ROC AUC": "{:.3f}",
        "Average Precision": "{:.3f}",
        "Precision @ 0.50": "{:.3f}"
    })
)

,Validation,ROC AUC,Average Precision,Precision @ 0.50
0,Before: client-grouped (seed=42),0.619,0.611,0.588
1,After: client-grouped (seed=2026),0.652,0.717,0.678


## 3. Leakage audit

## 3. Leakage audit

The leakage audit checks whether the final feature set contains the target itself, a target-derived field, or a feature that would only be available after the decision point.

The target is `is_declining_label`, derived from `trend_direction == "down"`.

I also check for suspicious feature names, duplicate content records, and client overlap between the validation partitions.

Because the starter dataset does not contain an observation date, this audit does not claim to establish time-based leakage. It checks leakage risks that can be identified from the available modeling table and validation setup.

In [16]:
# ============================================================
# SECTION 3 - LEAKAGE AUDIT
# ============================================================

# ------------------------------------------------------------
# 1. Target and explicitly known target-derived fields
# ------------------------------------------------------------

target_column = "is_declining_label"

known_leakage_candidates = [
    "trend_direction",
    "is_declining_label",
    "future_decline",
    "future_recovery",
    "target",
    "label",
    "priority_score",
    "action_type",
    "health_score",
    "refresh_tier"
]

present_known_candidates = [
    col
    for col in known_leakage_candidates
    if col in X.columns
]

print("Known target/leakage candidates present in X:")
print(present_known_candidates)

assert "trend_direction" not in X.columns, (
    "LEAKAGE: trend_direction is present in the feature matrix."
)

assert "is_declining_label" not in X.columns, (
    "LEAKAGE: target column is present in the feature matrix."
)

print("\nDirect target leakage check: PASSED")

Known target/leakage candidates present in X:
[]

Direct target leakage check: PASSED


In [17]:
# ============================================================
# 2. Suspicious feature-name audit
# ============================================================

suspicious_terms = [
    "target",
    "label",
    "future",
    "outcome",
    "decline",
    "recovery",
    "refresh",
    "priority",
    "action",
    "health_score"
]

suspicious_features = []

for feature in all_features:
    feature_lower = feature.lower()

    matched_terms = [
        term
        for term in suspicious_terms
        if term in feature_lower
    ]

    if matched_terms:
        suspicious_features.append(
            {
                "feature": feature,
                "matched_terms": ", ".join(matched_terms)
            }
        )

suspicious_features_df = pd.DataFrame(
    suspicious_features
)

print("Suspicious feature-name audit:")

if suspicious_features_df.empty:
    print("No suspicious feature names found.")
else:
    display(suspicious_features_df)

Suspicious feature-name audit:
No suspicious feature names found.


In [18]:
# ============================================================
# 3. Inspect the final feature set
# ============================================================

feature_audit = pd.DataFrame({
    "feature": all_features,
    "type": [
        "numeric" if col in numeric_features
        else "categorical"
        for col in all_features
    ]
})

display(feature_audit)

,feature,type
0,search_volume,numeric
1,competition,numeric
2,cpc,numeric
3,word_count,numeric
4,char_count,numeric
5,impressions_90d,numeric
6,clicks_90d,numeric
7,sessions_90d,numeric
8,ai_sessions_90d,numeric
9,days_with_impressions,numeric


In [19]:
# ============================================================
# 4. VALIDATION GROUP OVERLAP CHECK
# ============================================================

before_overlap = (
    set(groups_train_before)
    &
    set(groups_test_before)
)

after_overlap = (
    set(groups_train_after)
    &
    set(groups_test_after)
)

print("Before split client overlap:", len(before_overlap))
print("After split client overlap:", len(after_overlap))

assert len(before_overlap) == 0, (
    "LEAKAGE: clients overlap between before train and test sets."
)

assert len(after_overlap) == 0, (
    "LEAKAGE: clients overlap between after train and test sets."
)

print("\nClient separation check: PASSED")

Before split client overlap: 0
After split client overlap: 0

Client separation check: PASSED


In [20]:
# ============================================================
# 5. DUPLICATE CONTENT CHECK
# ============================================================

if "content_id" in model_df.columns:

    duplicate_content_rows = model_df["content_id"].duplicated().sum()

    print(
        "Duplicate content_id rows in modeling data:",
        duplicate_content_rows
    )

    if duplicate_content_rows == 0:
        print("Duplicate content check: PASSED")
    else:
        print(
            "Duplicate content check: duplicates remain. "
            "The modeling population should be reviewed."
        )

else:
    print(
        "content_id is not available, so an exact duplicate-content "
        "check cannot be performed."
    )

Duplicate content_id rows in modeling data: 0
Duplicate content check: PASSED


## 4. Claim rewrite

### Original bold claim

> The Random Forest accurately predicts which pages are declining and can identify pages that should be refreshed.

### Safer rewritten claim

> The Random Forest measured useful discrimination against the observed decline proxy in the validation data and can be used as a decision-support signal for prioritizing pages for further review.

This wording is narrower because the target is based on the observed `trend_direction` field rather than a confirmed future outcome. The model therefore provides a measured scoring signal against the available proxy label; it does not establish that a page will decline in the future or that refreshing a page will improve its performance.

In [21]:
# ============================================================
# SECTION 4 - CLAIM EVIDENCE
# ============================================================

claim_evidence = pd.DataFrame({
    "Metric": [
        "Before ROC AUC",
        "Before Average Precision",
        "After ROC AUC",
        "After Average Precision"
    ],
    "Value": [
        roc_before,
        ap_before,
        roc_after,
        ap_after
    ]
})

display(
    claim_evidence.style.format({
        "Value": "{:.3f}"
    })
)

,Metric,Value
0,Before ROC AUC,0.619
1,Before Average Precision,0.611
2,After ROC AUC,0.652
3,After Average Precision,0.717


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.